In [ ]:
import xarray as xr

import scipy
from scipy.stats import t

import pandas as pd
import xskillscore as xs
import numpy as np
import dask
import concurrent.futures
from concurrent.futures import ProcessPoolExecutor
from concurrent.futures import ThreadPoolExecutor
import logging

import albedo_functions as af

In [ ]:
# Configurazione logging
log_file = "process_log.txt"
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[
        logging.FileHandler(log_file),  # Write to file
    ]
)

In [ ]:
def dataset_season(dset,season):
    """
    SEASON SELECTION
    """
    if season == 'DJF':
        mon = [12,1,2]
    elif season == 'MAM':
        mon = [3,4,5]
    elif season == 'JJA':
        mon = [6,7,8]
    elif season == 'SON':
        mon = [9,10,11]
    elif season == 'Clim':
        mon = [11,12,1,2,3,4,5,6,7,8,9,10]
        
    dset = dset.sel(time=dset.time.dt.month.isin(mon))
    dset = dset.rolling(time = len(mon)).mean('time')
    dset = dset.sel(time = dset.time.dt.month == mon[-1])
    
    return dset

In [ ]:
def delta_sign(n1, slope1, se1, n2, slope2, se2):
    """
    Compare two regression slopes.

    Parameters:
        n1, n2 (int): Number of observations in each dataset
        slope1, slope2 (float): Slopes from the two regressions
        se1, se2 (float): Standard errors of the slopes

    Returns:
        float: Two-tailed p-value for the difference in slopes
    """
    df = n1 + n2 - 4
    t_stat = (slope1 - slope2) / (se1**2 + se2**2)**0.5
    p_value = 2 * (1 - t.cdf(abs(t_stat), df=df))
    return p_value

In [ ]:
exp_ctrl = 'a1ua'
exp_sens = 'a52o'
COV_PATH = '/confess/dicarlo/02-confess/05-confess-post-data/'
LAI_PATH = '/confess/dicarlo/00-dataset/02-confess/01-confess_data/'
SAVE_PATH = '/confess/dicarlo/cartella_ordinata/'

In [ ]:
# Open datasets with Dask chunking
cvh_sens = xr.open_dataset(COV_PATH + f'{exp_sens}_effective_cvh_199311-201910_1x1.nc', chunks=-1)['cvh']
cvl_sens = xr.open_dataset(COV_PATH + f'{exp_sens}_effective_cvl_199311-201910_1x1.nc', chunks=-1)['cvl']
lai_hv_a52o = xr.open_dataset(LAI_PATH + f'{exp_sens}/original_resolution/lai_hv/{exp_sens}_lai_hv_199311-201910_1x1.nc', chunks=-1)['LAI_HV']
lai_lv_a52o = xr.open_dataset(LAI_PATH + f'{exp_sens}/original_resolution/lai_lv/{exp_sens}_lai_lv_199311-201910_1x1.nc', chunks=-1)['LAI_LV']


cvh_ctrl = xr.open_dataset(COV_PATH + f'{exp_ctrl}_effective_cvh_1x1.nc', chunks=-1)['cvh']
cvl_ctrl = xr.open_dataset(COV_PATH + f'{exp_ctrl}_effective_cvl_1x1.nc', chunks=-1)['cvl']
a1ua_land_file = '/confess/dicarlo/00-dataset/02-confess/03-bc/'+exp_ctrl+'/icmcl_a1ua_regular_1x1.nc'
dset_a1ua = xr.open_dataset(a1ua_land_file, chunks=-1)
lai_hv_a1ua = dset_a1ua['lai_hv'].sel(time=slice(lai_hv_a52o.time[0],lai_hv_a52o.time[-1]))
lai_lv_a1ua = dset_a1ua['lai_lv'].sel(time=slice(lai_hv_a52o.time[0],lai_hv_a52o.time[-1]))

lai_ctrl = (lai_hv_a1ua * cvh_ctrl) + (lai_lv_a1ua * cvl_ctrl)
lai_sens = (lai_hv_a52o * cvh_sens) + (lai_lv_a52o * cvl_sens)
lai_ctrl['time'] = pd.to_datetime(lai_ctrl['time'].values).normalize().to_period('M').start_time
lai_sens['time'] = pd.to_datetime(lai_sens['time'].values).normalize().to_period('M').start_time

#select time from 1999
start_date = '1999-09-01'

lai_ctrl = lai_ctrl.sel(time=slice(start_date, None))
lai_sens = lai_sens.sel(time=slice(start_date, None))

In [ ]:
from scipy import stats
def trend(ds, time):
    slope, intercept, r_value, p_value, std_err = stats.linregress(time, ds)
    return slope, p_value, std_err

In [ ]:
def processing_std(ctrl, sens, save_path, season):
    logging.info(f'Starting {season}')
    try:
        # Extract seasonal data
        seasonal_sens = dataset_season(sens,season)
        seasonal_ctrl = dataset_season(ctrl,season)

        time_sens = xr.DataArray(
            np.arange(seasonal_sens.sizes["time"]),
            dims="time"
            )

        time_ctrl = xr.DataArray(
            np.arange(seasonal_ctrl.sizes["time"]),
            dims="time"
        )
    
        ctrl_slope, ctrl_p_value, ctrl_std_err = xr.apply_ufunc(
            trend,
            seasonal_ctrl.chunk(dict(time=-1)),
            time_ctrl,
            input_core_dims=[["time"], ["time"]],
            output_core_dims=[[], [], []],
            vectorize=True,
            dask="parallelized",
            output_dtypes=[float, float, float],
        )

        sens_slope, sens_p_value, sens_std_err = xr.apply_ufunc(
            trend,
            seasonal_sens.chunk(dict(time=-1)),
            time_sens,
            input_core_dims=[["time"], ["time"]],
            output_core_dims=[[],[], []],
            vectorize=True,
            dask="parallelized",
            output_dtypes=[float, float, float],
        )

        ctrl_slope.to_dataset(name='slope').to_netcdf(f"{SAVE_PATH}a1ua_lai_trend_{season}_1999.nc")
        ctrl_p_value.to_dataset(name='p').to_netcdf(f"{SAVE_PATH}a1ua_lai_trend_p_{season}_1999.nc")
        sens_slope.to_dataset(name='slope').to_netcdf(f"{SAVE_PATH}a52o_lai_trend_{season}_1999.nc")
        sens_p_value.to_dataset(name='p').to_netcdf(f"{SAVE_PATH}a52o_lai_trend_p_{season}_1999.nc")

        p_delta = delta_sign(len(seasonal_ctrl.time), ctrl_slope, ctrl_std_err, len(seasonal_sens.time), sens_slope, sens_std_err)

        p_delta = xr.DataArray(p_delta, 
                           name='p_delta',
                           dims=['lat', 'lon'],
                           coords={'lat': sens_slope.lat, 'lon': sens_slope.lon})

        p_delta.to_netcdf(f'{SAVE_PATH}delta_lai_trend_p_{season}_1999.nc')


    except Exception as e:
        logging.error(f"Failed for {season}: {e}", exc_info=True)

In [ ]:
%%time
seasons=['DJF']#, 'JJA', 'MAM', 'SON']
futures=[]
with concurrent.futures.ProcessPoolExecutor() as executor:
    futures = [executor.submit(processing_std, lai_ctrl, lai_sens, SAVE_PATH, i) for i in seasons]

    # Wait for all tasks to complete
    concurrent.futures.wait(futures)